# Preparation

## Gather Data

In [ ]:
# import standard modules
import pandas as pd
import seaborn as sns
import numpy as np
import matplotlib.pyplot as plt

# import modules
from sklearn.inspection import permutation_importance
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score, confusion_matrix, classification_report

# shut off some annoying warnings
import warnings
from pandas.core.common import SettingWithCopyWarning
warnings.simplefilter(action="ignore", category=SettingWithCopyWarning)

# show all columns
pd.set_option('display.max_columns', None)

# visualisation settings
pd.set_option('display.float_format', '{:,.2f}'.format)
sns.set_theme(style="whitegrid")

#warnungen
from sklearn.exceptions import DataConversionWarning, UndefinedMetricWarning
import warnings
warnings.filterwarnings(action='ignore', category=DataConversionWarning)
warnings.filterwarnings(action='ignore', category=UndefinedMetricWarning)
warnings.filterwarnings(action='ignore', category=DeprecationWarning)

In [ ]:
# read data
df = pd.read_csv('data_train.csv')

In [ ]:
#check data
df.head()

In [ ]:
# Notes: 33 columns, too match for Features

## EDA

In [ ]:
#check basic information
def overview(df):
    '''
    Create an overview of some key properties of a DataFrame's columns.
    VARs
        df: The DataFrame to be considered
    RETURNS:
        None
    '''
    df = df.copy()
    display(pd.DataFrame({'dtype': df.dtypes,  
                          'total': df.count(),  
                          'missing_n': df.isna().sum(),
                          'missing_%': df.isna().mean()*100, 
                          'uniques_n': df.nunique(), 
                          'uniques': [df[col].unique() for col in df.columns]   
                         }))
overview(df)

In [ ]:
# Notes

# 1. Nan: 
# Trim, WheelTypeID, WheelType, 
# PRIMEUNIT (95% missing), AUCGUART (95% missing) - drop
# Nan can be Impute: Color, Transmission, Nationality, Size, MMRA, VehBCost

# 2. Datatype: 
# datetime - PurchDate
# Keep as numeric: VehOdo, All MMR price columns, VehBCost, WarrantyCost, VehicleAge
# categorical: Auction (3 unique), VehYear (10 unique), Color (16 unique), Transmission (3 unique)
# categorical: WheelType (3 unique),  Nationality (4 unique), Size (12 unique)
# categorical: TopThreeAmericanName (4 unique), IsOnlineSale (0/1 → treat as categorical)
# High‑cardinality categorical columns (problematic for OHE): Model, SubModel, Trim, Make, BYRNO, VINZIP1, VNST

# 3. Drop: 
# BYRNO, Model & SubModel (very high cardinality), WheelTypeID (leave WheelType), VNST (leave VNZIP1), 
# WheelType (leave WheelTypeID), VehYear (leave VehicleAge), PRIMEUNIT & AUCGUART
# Trim (134 unique value, 3% missing) 
# Since only MMRCurrentRetailCleanPrice is better for used for modeling and visualizations,
# all other MMR fields were removed to simplify the dataset and avoid unnecessary noise

# 4. Neu Features: 
# PurchDate - PurchaseYear, PurchaseMonth, PurchaseDayOfWeek
# VehOdo - mileage per age
# Risk flags - IsOldCar (VehYear < threshold), IsVeryCheap (VehBCost < percentile), 
# IsSuspiciousPriceDrop (CurrentPrice << AcquisitionPrice)
# PriceGap = MMRCurrentRetailCleanPrice – VehBCost
# RelativePrice = VehBCost / MMRCurrentRetailCleanPrice
# IsHighMileage = VehOdo > percentile(75)
# NZIP1 - up to 3 characters

In [ ]:
# check distribution of values
df.describe().T

In [ ]:
# Notes
# VehOdo: wide spread (5k–115k miles)
# VehBCost: some extremely low values (min = 1 USD)
# # MMR price columns: min = 0.0 - placeholder for missing/invalid values, not real prices (not counted as NaN)
# IsOnlineSale: very low mean (3%), highly imbalanced binary feature

In [ ]:
# dublicate checken
df.duplicated().sum()

In [ ]:
# Notes
# no dublicate 

In [ ]:
# crosstab Target
target = df.loc[:, 'IsBadBuy']

crosstab = pd.crosstab(index=target, columns='count', normalize='columns')

fig, ax = plt.subplots(1, figsize=(9,7)) 
crosstab.plot(kind='pie', y='count', autopct='%1.1f%%', ax=ax)
ax.set(ylabel='', title = 'Verteilung: IsBadBuy')

In [ ]:
# Notes: 
# IsBadBuy - highly imbalanced (12% bad buys)

In [ ]:
#check correlations using Pearson Correlation
plt.figure(figsize=(8, 4))
cor = df.corr()
sns.heatmap(cor, annot=True, cmap=plt.cm.Reds)
plt.show()

In [ ]:
# Notes:
# MMR prices are highly redundant - reduce or transform
# VehYear & VehicleAge duplicate each other - keep one
# VehBCost & MMR prices (0,6- 0,8) - deviations signal bad buys 
# WarrantyCost & MMR prices (moderately) - WarrantyCost is essentially a proxy for vehicle value/condition
# IsBadBuy has weak correlations with all numeric features - the problem is non‑linear, boosting models + feature engineering 

In [ ]:
for col in ['VehOdo', 'VehBCost', 'WarrantyCost', 'MMRCurrentRetailCleanPrice']:
    plt.figure(figsize=(6, 2))
    sns.boxplot(x=df[col])
    plt.title(f'Boxplot: {col}')
    plt.show()

In [ ]:
# Notes:
# VehOdo: right-skewed, several high-mileage outliers (expected for used cars)
# VehBCost: right-skewed, extreme low outliers
# WarrantyCost: right-skewed, long tail, but no extreme anomalies

In [ ]:
sns.scatterplot(x='VehBCost', y='MMRCurrentRetailCleanPrice', data=df)

In [ ]:
# Notes:
# VehBCost vs MMRCurrentRetailCleanPrice: strong positive relationship, 
# confirming that acquisition cost aligns with market valuation; no unexpected anomalies

In [ ]:
# pairplot
key_cols = ['VehOdo', 'VehBCost', 'WarrantyCost', 'MMRCurrentRetailCleanPrice', 'VehicleAge']
sns.pairplot(df[key_cols])

In [ ]:
# Notes:
# VehBCost and MMRCurrentRetailCleanPrice - strong positive relationship

In [ ]:
# buyer analyse
buyer_counts = df['BYRNO'].value_counts()
print(buyer_counts.describe())
sns.histplot(buyer_counts, bins=20)
plt.title("Distribution of number of cars purchased per buyer")

In [ ]:
# Notes
# most buyers bought one or a few cars, while others bought a lot of cars;'BYRNO' can be dropped